In [2]:
# directories
pdfs_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\pdfs'
ocr_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\ocr_text'
extracted_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\extracted_csv'

In [3]:
# 1869 data extraction

import re
import csv
from pathlib import Path
import time

def clean_text(text):
    """Clean OCR artifacts and normalize text."""
    replacements = {
        'Î': 'A', 'Î•': 'E', 'Îœ': 'M', 'Î': 'N', 'Ð¡': 'C', 'Ð¢': 'T',
        'Ã‰': 'E', 'Ñ': 'c', 'Ðµ': 'e', 'Ñ€': 'p', 'Ð': 'N',
        '`': "'", "'": "'", '"': '"', '"': '"',
        '\xad': '', '­': '',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def normalize_for_matching(text):
    """Normalize text for pattern matching - removes extra spaces."""
    return re.sub(r'\s+', ' ', text)

def normalize_editor_publisher_text(text):
    """
    Normalize text specifically for editor/publisher extraction.
    Handles OCR artifacts like hyphenated line breaks and missing spaces.
    """
    normalized = text
    
    # Remove hyphenated line breaks (e.g., "edi- tors" -> "editors", "pub- lisher" -> "publisher")
    normalized = re.sub(r'-\s+', '', normalized)
    
    # Fix common OCR run-together patterns
    normalized = re.sub(r'(editors?)(and)(pub)', r'\1 \2 \3', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(editors?)(and)(prop)', r'\1 \2 \3', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(publishers?)(and)(prop)', r'\1 \2 \3', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(publishers?)(and)', r'\1 \2', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(proprietors?)(and)', r'\1 \2', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(and)(publishers?)', r'\1 \2', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(and)(proprietors?)', r'\1 \2', normalized, flags=re.IGNORECASE)
    
    # Normalize multiple spaces
    normalized = re.sub(r'\s+', ' ', normalized)
    
    return normalized

def extract_circulation(text):
    """Extract circulation number from text."""
    patterns = [
        r'circulation[:\s]+(?:about\s+)?(\d[\d,\.]+)',
        r'claims?\s+(?:about\s+)?(\d[\d,\.]+)\s+circulation',
        r'circ(?:ulation|\'?l?n)[:\s\.]+(?:about\s+)?(\d[\d,\.]+)',
        r'(\d[\d,\.]+)\s+circ(?:ulation|\'?l?n)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).replace(',', '').replace('.', '')
    return ''

def extract_political_affiliation(text):
    """Extract political affiliation from text."""
    affiliations = ['democratic', 'republican', 'independent', 'neutral',
                    'whig', 'conservative', 'liberal', 'radical']
    text_lower = text.lower()
    for affiliation in affiliations:
        if re.search(rf';\s*{affiliation}\b', text_lower):
            return affiliation.capitalize()
    return ''

def extract_subscription_details(text):
    """Extract detailed subscription info."""
    daily_match = re.search(r'subscription[-\s]+daily\s+\$(\d+(?:\s+\d{2})?)', text, re.IGNORECASE)
    if daily_match:
        return f"${daily_match.group(1).replace(' ', '.')}"
    
    weekly_match = re.search(r'(?:subscription[-\s]+)?weekly\s+\$(\d+(?:\s+\d{2})?)', text, re.IGNORECASE)
    if weekly_match:
        return f"${weekly_match.group(1).replace(' ', '.')}"
    
    std_match = re.search(r'subscription[:\s]+\$(\d+(?:\s+\d{2})?)', text, re.IGNORECASE)
    if std_match:
        return f"${std_match.group(1).replace(' ', '.')}"
    
    cents_match = re.search(r'subscription\s+(\d+)\s+cents', text, re.IGNORECASE)
    if cents_match:
        return f"${int(cents_match.group(1))/100:.2f}"
    
    return ''

def extract_frequency(text):
    """Extract publication frequency from text."""
    text_lower = text.lower()
    
    if re.search(r'every\s+(?:morning|evening|day)', text_lower):
        return 'Daily & Weekly' if 'and weekly' in text_lower or 'weekly,' in text_lower else 'Daily'
    if re.search(r'tri-?weekly', text_lower):
        return 'Tri-weekly & Weekly' if 'and weekly' in text_lower else 'Tri-weekly'
    if re.search(r'semi-?weekly', text_lower):
        return 'Semi-weekly & Weekly' if 'and weekly' in text_lower else 'Semi-weekly'
    if re.search(r'semi-?monthly', text_lower):
        return 'Semi-monthly'
    if 'quarterly' in text_lower:
        return 'Quarterly'
    if 'monthly' in text_lower:
        return 'Monthly'
    
    days = ['sundays', 'mondays', 'tuesdays', 'wednesdays', 'thursdays', 'fridays', 'saturdays']
    for day in days:
        if day in text_lower:
            return 'Weekly'
    return ''

def extract_established(text):
    """Extract establishment year from text."""
    patterns = [
        r'establish[e]?d\s+(\d{4})',
        r'estab[-\s]*lished\s+(\d{4})',
        r'es[-\s]*tablished\s+(\d{4})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            year = match.group(1)
            if 1700 <= int(year) <= 1900:
                return year
    return ''

def clean_name(name):
    """Clean and validate an extracted name."""
    if not name:
        return None
    
    name = name.strip().strip(',;:.')
    
    if len(name) < 3:
        return None
    
    false_positives = ['Four', 'Eight', 'The', 'And', 'Weekly', 'Daily', 'Semi', 
                       'Tri', 'Monthly', 'Sunday', 'Saturday', 'Friday', 'Thursday',
                       'Wednesday', 'Tuesday', 'Monday', 'About', 'Claims', 'Size',
                       'Subscription', 'Established', 'Circulation', 'Pages',
                       'Democratic', 'Republican', 'Independent', 'Neutral',
                       'Temperance', 'Association']
    if name in false_positives:
        return None
    
    if name.replace(',', '').replace('.', '').isdigit():
        return None
    
    if re.search(r'\b(editor|publisher|proprietor|and)\s*$', name, re.IGNORECASE):
        return None
    
    if name[0].islower():
        return None
    
    return name

def add_name_if_unique(name, name_list):
    """Add a name to the list if it's not a duplicate."""
    cleaned = clean_name(name)
    if not cleaned:
        return False
    
    cleaned_lower = cleaned.lower()
    
    for existing in name_list:
        if cleaned_lower == existing.lower() or cleaned_lower in existing.lower():
            return False
    
    to_remove = [e for e in name_list if e.lower() in cleaned_lower]
    for item in to_remove:
        name_list.remove(item)
    
    name_list.append(cleaned)
    return True

def extract_editor_publisher(text):
    """Extract editor and publisher names from text."""
    editors, publishers = [], []
    
    normalized = normalize_editor_publisher_text(text)
    normalized = normalize_for_matching(normalized)
    
    normalized = re.sub(r'(\w)(and)(\w)', r'\1 \2 \3', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(editor[s]?)(and)', r'\1 \2', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(and)(pub)', r'\1 \2', normalized, flags=re.IGNORECASE)
    normalized = re.sub(r'(and)(prop)', r'\1 \2', normalized, flags=re.IGNORECASE)
    
    # Split by semicolons to process each segment separately
    segments = re.split(r';', normalized)
    
    for segment in segments:
        segment = segment.strip()
        if not segment:
            continue
        
        name_pattern = r'([A-Z][A-Za-z\.\s&,]+?)'
        
        combined_patterns = [
            re.compile(name_pattern + r',?\s+editors?\s+and\s+publishers?', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+editors?\s+and\s+proprietors?', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+editors?andpublishers?', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+editors?andproprietors?', re.IGNORECASE),
        ]
        
        editor_patterns = [
            re.compile(name_pattern + r',?\s+editors?\s*$', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+editors?\s*,', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+editor-in-chief', re.IGNORECASE),
        ]
        
        publisher_patterns = [
            re.compile(name_pattern + r',?\s+publishers?\s*$', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+proprietors?\s*$', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+publishers?\s+and\s+proprietors?', re.IGNORECASE),
            re.compile(name_pattern + r',?\s+publishersandproprietors?', re.IGNORECASE),
        ]
        
        matched_combined = False
        for pattern in combined_patterns:
            match = pattern.search(segment)
            if match:
                add_name_if_unique(match.group(1), editors)
                add_name_if_unique(match.group(1), publishers)
                matched_combined = True
                break
        
        if matched_combined:
            continue
        
        for pattern in editor_patterns:
            match = pattern.search(segment)
            if match:
                add_name_if_unique(match.group(1), editors)
                break
        
        for pattern in publisher_patterns:
            match = pattern.search(segment)
            if match:
                add_name_if_unique(match.group(1), publishers)
                break
    
    return {'editor': '; '.join(editors), 'publisher': '; '.join(publishers)}

def is_state_header(line):
    """Check if a line is a state header (e.g., 'ALABAMA.' or 'ALABAMA .')"""
    # List of valid US states and territories for the period
    valid_states = [
        'ALABAMA', 'ARKANSAS', 'ARIZONA', 'CALIFORNIA', 'COLORADO', 
        'CONNECTICUT', 'DAKOTA', 'DELAWARE', 'DISTRICT OF COLUMBIA',
        'FLORIDA', 'GEORGIA', 'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA',
        'KANSAS', 'KENTUCKY', 'LOUISIANA', 'MAINE', 'MARYLAND',
        'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA', 'MISSISSIPPI',
        'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA', 'NEW HAMPSHIRE',
        'NEW JERSEY', 'NEW MEXICO', 'NEW YORK', 'NORTH CAROLINA',
        'OHIO', 'OREGON', 'PENNSYLVANIA', 'RHODE ISLAND',
        'SOUTH CAROLINA', 'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT',
        'VIRGINIA', 'WASHINGTON', 'WEST VIRGINIA', 'WISCONSIN', 'WYOMING',
        # Territories and other regions
        'INDIAN TERRITORY', 'DOMINION OF CANADA', 'BRITISH COLONIES',
        'CANADA', 'NEWFOUNDLAND', 'NOVA SCOTIA', 'NEW BRUNSWICK',
        'PRINCE EDWARD ISLAND', 'MANITOBA', 'ONTARIO', 'QUEBEC',
        'BRITISH COLUMBIA'
    ]
    
    # Clean the line: remove periods, extra spaces, and strip
    # This handles both "ALABAMA." and "ALABAMA ." patterns
    cleaned = re.sub(r'\s*\.\s*', '', line).strip().upper()
    
    # Check if it matches a state name
    if cleaned in valid_states:
        return cleaned
    
    return None

def is_valid_town_name(town):
    """Check if the extracted town name is a valid town (not an index/header entry)."""
    invalid_patterns = [
        r'^A\s+LIST', r'DOMINION', r'CANADA', r'BRITISH', r'COLONIES',
        r'UNITED\s+STATES', r'TERRITORIES', r'^INDEX', r'^PAGE\s*\d*',
        r'NEWSPAPERS?', r'PERIODICALS?', r'ALPHABETICALLY', r'ARRANGED',
        r'GIVING\s+NAME', r'DAYS\s+OF\s+ISSUE', r'SUBSCRIPTION\s+PRICE',
        r'EDITOR.?S?\s+AND\s+PUBLISHER', r'CIRCULATION', r'ADVERTISEMENTS?',
        r'PRINTING\s+MATERIAL', r'IN\s+WHICH', r'ARE\s+PUBLISHED',
        r'^NOTE', r'^\d+$', r'^THE\s+', r'ALIST\s+OF',
    ]
    
    town_upper = town.upper().strip()
    for pattern in invalid_patterns:
        if re.search(pattern, town_upper):
            return False
    
    if len(town) > 50 or len(town.split()) > 4:
        return False
    
    return True

def is_valid_entry_text(entry_text):
    """Check if the entry text looks like a valid newspaper entry."""
    invalid_patterns = [
        r'ALPHABETICALLY\s+BY\s+TOWNS', r'DAYS\s+OF\s+ISSUE',
        r'POLITICS\s+OR\s+GENERAL\s+CHARACTER', r'DATE\s+OF\s+ESTABLISHMENT',
        r'EDITOR.?S?\s+AND\s+PUBLISHER.?S?\s+NAMES', r'GIV-?\s*ING\s+NAME',
        r'DOMINION\s+OF\s+CANADA', r'BRITISH\s+COLONIES',
        r'UNITED\s+STATES\s+AND\s+TERRITORIES',
        r'A\s*LIST\s+OF\s+THE\s+NEWSPAPERS',
    ]
    
    text_upper = entry_text.upper()
    for pattern in invalid_patterns:
        if re.search(pattern, text_upper):
            return False
    return True

def parse_newspaper_entries(text):
    """Parse the text into individual newspaper entries with state tracking."""
    text = clean_text(text)
    lines = text.split('\n')
    entries = []
    current_entry = []
    current_town = None
    current_state = None
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Check if this line is a state header
        state_match = is_state_header(line)
        if state_match:
            # Update state if it's different from current (handles repeated page headers)
            if state_match != current_state:
                # Save any pending entry before changing state
                if current_entry and current_town:
                    entries.append((current_state, current_town, ' '.join(current_entry)))
                current_state = state_match
                current_entry = []
                current_town = None
            # Skip the state header line either way
            continue
        
        new_entry_match = re.match(
            r'^([A-Z][A-Z\s,\.]+?)(?:,\s*|\s+)([A-Z][a-zA-Z\s&\'\.\-]+?)\s*[;:]',
            line
        )
        
        if new_entry_match:
            if current_entry and current_town:
                entries.append((current_state, current_town, ' '.join(current_entry)))
            current_town = new_entry_match.group(1).strip().rstrip(',')
            current_entry = [line]
        elif current_entry:
            current_entry.append(line)
    
    if current_entry and current_town:
        entries.append((current_state, current_town, ' '.join(current_entry)))
    
    return entries

def parse_entry_details(state, town, entry_text):
    """Extract structured data from a single entry."""
    result = {
        'state': state if state else '',
        'town': town.strip().title(),
        'newspaper_name': '',
        'frequency': extract_frequency(entry_text),
        'political_affiliation': extract_political_affiliation(entry_text),
        'subscription_price': extract_subscription_details(entry_text),
        'established': extract_established(entry_text),
        'circulation': extract_circulation(entry_text),
        'raw_text': entry_text[:300] + '...' if len(entry_text) > 300 else entry_text
    }
    
    name_match = re.match(
        rf'^{re.escape(town)}[,\s]+([A-Za-z][A-Za-z\s&\'\.\-,]+?)\s*[;:]',
        entry_text, re.IGNORECASE
    )
    if name_match:
        result['newspaper_name'] = name_match.group(1).strip().rstrip(',;:')
    
    people = extract_editor_publisher(entry_text)
    result['editor'] = people['editor']
    result['publisher'] = people['publisher']
    
    return result

def process_file(input_path, output_path=None):
    """Process the input file and write results to CSV."""
    start_time = time.time()
    
    print(f"Reading file: {input_path}")
    with open(input_path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()
    
    print("Parsing entries...")
    raw_entries = parse_newspaper_entries(text)
    total_entries = len(raw_entries)
    print(f"Found {total_entries} raw entries")
    
    results = []
    
    for i, (state, town, entry_text) in enumerate(raw_entries):
        if (i + 1) % 100 == 0 or i == total_entries - 1:
            elapsed = time.time() - start_time
            pct = (i + 1) / total_entries * 100
            print(f"Processing: {i + 1}/{total_entries} ({pct:.1f}%) - {elapsed:.1f}s elapsed")
        
        if not is_valid_town_name(town):
            continue
        if not is_valid_entry_text(entry_text):
            continue
        
        details = parse_entry_details(state, town, entry_text)
        if details['newspaper_name'] and len(details['newspaper_name']) > 1:
            results.append(details)
    
    if output_path is None:
        output_path = Path(input_path).stem + '_extracted.csv'
    
    fieldnames = ['state', 'town', 'newspaper_name', 'frequency', 'political_affiliation', 
                  'subscription_price', 'established', 'editor', 'publisher',
                  'circulation', 'raw_text']
    
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    elapsed = time.time() - start_time
    print(f"\n{'='*50}")
    print(f"Completed in {elapsed:.1f} seconds")
    print(f"Processed {len(results)} valid entries")
    print(f"Output written to: {output_path}")
    print(f"{'='*50}")
    print(f"Entries with state: {sum(1 for r in results if r['state'])}")
    print(f"Entries with frequency: {sum(1 for r in results if r['frequency'])}")
    print(f"Entries with political affiliation: {sum(1 for r in results if r['political_affiliation'])}")
    print(f"Entries with subscription price: {sum(1 for r in results if r['subscription_price'])}")
    print(f"Entries with established date: {sum(1 for r in results if r['established'])}")
    print(f"Entries with editor: {sum(1 for r in results if r['editor'])}")
    print(f"Entries with publisher: {sum(1 for r in results if r['publisher'])}")
    print(f"Entries with circulation: {sum(1 for r in results if r['circulation'])}")
    
    return results

# USAGE

import os
for file in os.listdir(ocr_text_dir)[:1]:
    input = os.path.join(ocr_text_dir, file)
    output = os.path.join(extracted_text_dir, file[:-3] + 'csv')
    results = process_file(input, output)

Reading file: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1869.txt
Parsing entries...
Found 3079 raw entries
Processing: 100/3079 (3.2%) - 0.5s elapsed
Processing: 200/3079 (6.5%) - 0.7s elapsed
Processing: 300/3079 (9.7%) - 1.0s elapsed
Processing: 400/3079 (13.0%) - 1.2s elapsed
Processing: 500/3079 (16.2%) - 1.3s elapsed
Processing: 600/3079 (19.5%) - 1.5s elapsed
Processing: 700/3079 (22.7%) - 1.7s elapsed
Processing: 800/3079 (26.0%) - 1.8s elapsed
Processing: 900/3079 (29.2%) - 2.0s elapsed
Processing: 1000/3079 (32.5%) - 2.1s elapsed
Processing: 1100/3079 (35.7%) - 2.3s elapsed
Processing: 1200/3079 (39.0%) - 2.4s elapsed
Processing: 1300/3079 (42.2%) - 2.6s elapsed
Processing: 1400/3079 (45.5%) - 2.7s elapsed
Processing: 1500/3079 (48.7%) - 2.9s elapsed
Processing: 1600/3079 (52.0%) - 3.0s elapsed
Processing: 1700/3079 (55.2%) - 3.2s elapsed
Processing: 1800/3079 (58.5%) - 3.5s elapsed
Processing: 1900/3079 (61.7%) - 3.7s elapsed
Processi